In [10]:
%load_ext autoreload
%autoreload 2

import os
import sys

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Set rendering backend based on platform
# macOS: use glfw (native OpenGL)
# Linux: use osmesa (software rendering) or egl (headless GPU)
if sys.platform == "darwin":
    os.environ["MUJOCO_GL"] = "glfw"
    print("Using macOS with GLFW rendering")
else:
    # Change this to egl if GPU is available on Linux
    os.environ["MUJOCO_GL"] = "egl"
    print("Using Linux with osmesa rendering")

import mediapy as media
from pathlib import Path
import imageio

from track_mjx.agent import checkpointing
from track_mjx.analysis import rollout, render
from track_mjx import utils

from vnl_mjx.tasks.rodent.imitation import Imitation
from vnl_mjx.tasks.rodent import consts as rodent_consts
from ml_collections import config_dict

from omegaconf import DictConfig, OmegaConf

import huggingface_hub as hf_hub

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using Linux with osmesa rendering


In [2]:
hf_checkpoint_path = "rodent/rodent-analysis/251006_144548_202519"
model_local_dir = Path.cwd().parent / "model_checkpoints"
# Download model from model repo
model_download_dir = hf_hub.snapshot_download(
    repo_id="talmolab/MIMIC-MJX", 
    repo_type="model", # download from model repo
    allow_patterns=hf_checkpoint_path + "/*", # path with model id
    local_dir=model_local_dir
)
print(f"Downloaded model to {model_download_dir}")

/home/mila/a/aidan.sirbu/track-mjx/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 22 files: 100%|██████████| 22/22 [00:00<00:00, 742.76it/s]

Downloaded model to /home/mila/a/aidan.sirbu/track-mjx/model_checkpoints


In [3]:
hf_data_path = "data/rodent/rodent_reference_clips.h5"
# Download data from dataset repo
data_download_dir = hf_hub.hf_hub_download(
    repo_id="talmolab/MIMIC-MJX", 
    repo_type="dataset", # download from dataset repo
    filename=hf_data_path, # dataset name
    local_dir=Path.cwd().parent
)
print(f"Downloaded data to {data_download_dir}")

Downloaded data to /home/mila/a/aidan.sirbu/track-mjx/data/rodent/rodent_reference_clips.h5


In [4]:
# replace with your checkpoint path
ckpt_path = model_local_dir / hf_checkpoint_path

ckpt = checkpointing.load_checkpoint_for_eval(ckpt_path)
cfg = ckpt["cfg"]

# Update the data path to the downloaded data
cfg.data_path = Path(data_download_dir)

Loading checkpoint from /home/mila/a/aidan.sirbu/track-mjx/model_checkpoints/rodent/rodent-analysis/251006_144548_202519 at step 99


Aidan Edits

In [5]:
print(OmegaConf.to_yaml(cfg))

data_path: !!python/object/apply:pathlib.PosixPath
- /
- home
- mila
- a
- aidan.sirbu
- track-mjx
- data
- rodent
- rodent_reference_clips.h5
env_config:
  env_name: rodent_multi_clip
  task_name: imitation
  walker_name: rodent
  render_camera_name: close_profile
  render_fps: 50
  render_interval: 4
  env_args:
    solver: cg
    iterations: 5
    ls_iterations: 5
    physics_steps_per_control_step: 5
    reset_noise_scale: 0.001
    mj_model_timestep: 0.002
    mocap_hz: 50
  reward_weights:
    too_far_dist: 0.01
    bad_pose_dist: 20
    bad_quat_dist: 1
    ctrl_cost_weight: 0.02
    ctrl_diff_cost_weight: 0.02
    energy_cost_weight: 0.01
    pos_reward_weight: 1
    quat_reward_weight: 1
    joint_reward_weight: 1
    angvel_reward_weight: 0.0
    bodypos_reward_weight: 0.0
    endeff_reward_weight: 1
    healthy_z_range:
    - 0.0325
    - 0.5
    pos_reward_exp_scale: 400.0
    quat_reward_exp_scale: 4.0
    joint_reward_exp_scale: 0.25
    angvel_reward_exp_scale: 0.5
    b

In [6]:
ckpt = checkpointing.load_checkpoint_for_eval("/home/mila/a/aidan.sirbu/scratch/track-mjx/model_checkpoints/251103_153351_658864")

Loading checkpoint from /home/mila/a/aidan.sirbu/scratch/track-mjx/model_checkpoints/251103_153351_658864 at step 99


In [8]:
cfg = ckpt['cfg']

env = rollout.create_environment(cfg)
inference_fn = checkpointing.load_inference_fn(cfg, ckpt["policy"])
generate_rollout = rollout.create_rollout_generator(
    cfg, 
    env, 
    inference_fn, 
    log_activations=False, 
    log_metrics=False, 
    log_sensor_data=False
)

In [ ]:
single_rollout = generate_rollout(clip_idx=1)

In [12]:
with imageio.get_writer('rollout.mp4', fps=cfg.render_config.render_fps) as writer:
    video = env.render(
        single_rollout['rollout_states'],
        camera=f"{cfg.render_config.render_camera_name}-ghost",
        height=480,
        width=640,
    )
    for frame in video:
        writer.append_data(frame)

media.show_video(video)

/home/mila/a/aidan.sirbu/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(


This browser does not support the video tag.


In [35]:
frames, realtime_framerate = render.render_rollout(
    cfg, 
    single_rollout, 
    height=480,
    width=640,
)

# save the video to disk
# media.write_video(Path(ckpt_path) / "rollout.mp4", frames, fps=realtime_framerate)
media.show_video(frames, fps=realtime_framerate)

MuJoCo Rendering with Ghost Model...
Rendering every 1 steps; realtime fps: 100


100%|██████████| 500/500 [00:02<00:00, 219.25it/s]


This browser does not support the video tag.


End Aidan Edits

In [7]:
#TEMP: add max_start_frames to cfg to support older checkpoints
cfg.env_config.env_args.max_start_frame = 40

In [8]:
env = rollout.create_environment(cfg)
inference_fn = checkpointing.load_inference_fn(cfg, ckpt["policy"])
generate_rollout = rollout.create_rollout_generator(
    cfg, 
    env, 
    inference_fn, 
    log_activations=False, 
    log_metrics=False, 
    log_sensor_data=False
)

Converting to torque actuators
Rescaling body tree with scale factor 0.9


/home/mila/a/aidan.sirbu/.conda/envs/vnl_playground/lib/python3.11/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [9]:
single_rollout = generate_rollout(clip_idx=0)

In [10]:
frames, realtime_framerate = render.render_rollout(
    cfg, 
    single_rollout, 
    height=480,
    width=640,
)

# save the video to disk
media.write_video(Path(ckpt_path) / "rollout.mp4", frames, fps=realtime_framerate)
media.show_video(frames, fps=realtime_framerate)

ConfigAttributeError: Missing key walker_name
    full_key: walker_config.walker_name
    object_type=dict